# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [1]:
# Only needed for Udacity workspace

# import importlib.util
# import sys

# # Check if 'pysqlite3' is available before importing
# if importlib.util.find_spec("pysqlite3") is not None:
#     import pysqlite3
#     sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool, Tool

from dotenv import load_dotenv
from typing import List, Dict
import chromadb

from tavily import TavilyClient

from pydantic import BaseModel, Field

In [3]:
load_dotenv()

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [4]:
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")

In [5]:
queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
    "When was EA Sports FC 25 released?",
    "Was there any Halo Infinite released in 2021?",
]

In [6]:
@tool

def retrieve_game(query: str, n:int=1) -> List[Dict]:
    """
    Find most relevant information from the Vector DB
    """

    results = collection.query(
        query_texts=[query],
        n_results=n
    )

    docs = []

    for metadata, document in zip(results["metadatas"][0], results["documents"][0]):
        docs.append({
            "Description": metadata["Description"],
            "Genre": metadata["Genre"],
            "Name": metadata["Name"],
            "Publisher": metadata["Publisher"],
            "YearOfRelease": metadata["YearOfRelease"],
            "Platform": metadata["Platform"]
            # "content": document
        })

    return docs

In [7]:
q = "Which one was the first 3D platformer Mario game?"
retrieved_docs = retrieve_game(q)
print(retrieved_docs)

[{'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", 'Genre': 'Platformer', 'Name': 'Super Mario 64', 'Publisher': 'Nintendo', 'YearOfRelease': 1996, 'Platform': 'Nintendo 64'}]


In [8]:
doc_str = ""
for doc in range(len(retrieved_docs)):
    doc_str = doc_str + f"{retrieved_docs[doc]["Name"]}: {retrieved_docs[doc]["Description"]}" + "\n"
doc_str

"Super Mario 64: A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.\n"

#### Evaluate Retrieval Tool

In [9]:
class EvaluateReport(BaseModel):
    useful: bool = Field(description="x")
    description: str = Field(description="y")
    
@tool
def evaluate_retrieval(
    question: str, 
    retrieved_docs: List[Dict]) -> Dict:
    """
    Tool to evaluate the quality of retrieved documents and check if it is useful to answer the query.
    
    args: 
    - question - str: user question
    - retrieved_docs - List[Dict]: retrieved documents closest to the user query found in the Vector DB

    returns:
    - useful: bool
    - description: str
    """
   
    if not retrieved_docs:
        return {
            "useful": False,
            "description": "No document retrieved from Vector DB."
        }

    doc_str = ""
    for doc in range(len(retrieved_docs)):
        doc_str = doc_str + f"{retrieved_docs[doc]["Name"]}: {retrieved_docs[doc]["Description"]}" + "\n"
    
    prompt = f"""
        Your task is to evaluate if the documents are enough to respond the query.
        Give a detailed explanation, so it's possible to take an action to accept it or not.

        query = {question}
        Documents = {doc_str}

        Evaluate these documents contain sufficient information to answer the question.    
    """
        
    # Invoke the LLM
    llm = LLM(model= "gpt-4o-mini", temperature=0)
    response = llm.invoke(input=prompt, response_format=EvaluateReport)
    
    return response.content

In [10]:
reponse_evaluated = evaluate_retrieval(q, retrieved_docs)
print(q)
print(reponse_evaluated)

Which one was the first 3D platformer Mario game?
{"useful":true,"description":"The document provides sufficient information to answer the query regarding the first 3D platformer Mario game. It specifically mentions 'Super Mario 64' as a groundbreaking 3D platformer, which directly addresses the question. Additionally, it highlights the significance of the game in setting new standards for the genre, reinforcing its importance as the first of its kind for the Mario franchise. Therefore, the document is adequate to respond to the query."}


#### Game Web Search Tool

In [11]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(question: str) -> List[Dict]:
    """
    Semantic web search using Tavily.

    args:
    - question: a question about game industry
    """

    response = tavily_client.search(
        query=question,
        max_results=3
    )

    return response.get("results", [])

game_web_search("When was EA Sports FC 25 released?")

[{'url': 'https://easportsfc.fandom.com/wiki/EA_Sports_FC_25',
  'title': 'EA Sports FC 25',
  'content': '# EA Sports FC 25. ***EA Sports FC 25*** is a football game developed by EA Canada and EA Romania, it will be released on September 27 2024, it will be the 2nd game to be under the title EA Sports FC and the 32nd overall installment of the series, It will release for PlayStation 4, PlayStation 5, Xbox One, Xbox Series X/S, PC, and Nintendo Switch. :   *Main article: List of teams in EA Sports FC 25*. EA SPORTS FC 25 - Official Reveal Trailer. EA SPORTS FC 25 - Official Gameplay Deep Dive. EA SPORTS FC 25 - 5v5 Rush Deep Dive. EA SPORTS FC 25 - Official Ultimate Team Deep Dive. EA SPORTS FC 25 - Official Career Deep Dive. EA SPORTS FC 25 Official Launch Trailer - For The Club. | FIFA Online\xa0·\xa0FIFA Online 2\xa0·\xa0FIFA Online 3\xa0·\xa0FC Online 4\xa0·\xa0FIFA Superstars\xa0·\xa0FIFA World\xa0·\xa0EA Sports FC Mobile |.',
  'score': 0.9302622,
  'raw_content': None},
 {'url':

### Agent

In [12]:
agent_instructions = (
        "You are Udaplay, an AI assistant specialized in video games.\n"
        "Use available tools to answer questions with verified information.\n"
        "Use `retrieve_game` tool to search for available information.\n"
        "Always use `evaluate_retrieval` tool to assess the usefulness (True or False) of retrieved documents. \n"
        "Provide the evaluate_retrieval tool with the user's question and the documents retrieved from retrieve_game.\n"
        "The evaluate_retrieval tool returns an EvaluationReport with two fields:\n"
        "  - 'useful': a boolean indicating if the documents are sufficient (True/False)\n"
        "  - 'description': detailed explanation of the evaluation result\n"
        "If the 'useful' field is False, use the web_search tool to find additional information.\n"
        "If the 'useful' field is True, use the documents from retrieve_game to answer the question directly.\n"
        "Always provide helpful and accurate responses based on the evaluation results."
    )

agent = Agent(
    model_name="gpt-4o-mini",
    instructions= agent_instructions,
        tools=[retrieve_game, evaluate_retrieval, game_web_search]

)


In [13]:
q = "When Pokémon Gold and Silver was released?"
response = agent.invoke(query = q)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__


In [14]:
print(response.get_final_state().keys())

dict_keys(['user_query', 'instructions', 'messages', 'current_tool_calls', 'session_id', 'total_tokens'])


In [15]:
def print_run_object(response):
    final_state = response.get_final_state()
    final_answer = final_state["messages"][-1]
    print("*** Final Answer ***:", final_answer)
    print("*** Query: ***", final_state["user_query"])
    print("*** Tools: ***", final_state["current_tool_calls"])
    print(final_answer)
    print("-"*132)

In [16]:
print_run_object(response)

*** Final Answer ***: role='assistant' content='Pokémon Gold and Silver were released in Japan on November 21, 1999. They later saw a release in North America on October 15, 2000. These games were developed by Game Freak and published by Nintendo for the Game Boy Color.' tool_calls=None token_usage=None
*** Query: *** When Pokémon Gold and Silver was released?
*** Tools: *** None
role='assistant' content='Pokémon Gold and Silver were released in Japan on November 21, 1999. They later saw a release in North America on October 15, 2000. These games were developed by Game Freak and published by Nintendo for the Game Boy Color.' tool_calls=None token_usage=None
------------------------------------------------------------------------------------------------------------------------------------


In [17]:
queries = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for Playstation 5?",
    "When was EA Sports FC 25 released?",
    "Was there any Halo Infinite released in 2021?",
]

responses = []

for q in queries:
    response = agent.invoke(query = q, session_id="game_research_agent_session")
    responses.append(response)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor


In [20]:
for response in responses:
    messages = response.get_final_state()["messages"]
    current_tool_calls = response.get_final_state()["current_tool_calls"]
    print(f" ✅ RESPONSE")
    for msg in messages:
        print(f""" ➡️➡️ (role = {msg.role},
                   ➡️ content = {msg.content},
                   ➡️ tool_calls = {getattr(msg, 'tool_calls', None)})""")
        print("\n" + "-"*125 + "\n")

 ✅ RESPONSE
 ➡️➡️ (role = system,
                   ➡️ content = You are Udaplay, an AI assistant specialized in video games.
Use available tools to answer questions with verified information.
Use `retrieve_game` tool to search for available information.
Always use `evaluate_retrieval` tool to assess the usefulness (True or False) of retrieved documents. 
Provide the evaluate_retrieval tool with the user's question and the documents retrieved from retrieve_game.
The evaluate_retrieval tool returns an EvaluationReport with two fields:
  - 'useful': a boolean indicating if the documents are sufficient (True/False)
  - 'description': detailed explanation of the evaluation result
If the 'useful' field is False, use the web_search tool to find additional information.
If the 'useful' field is True, use the documents from retrieve_game to answer the question directly.
Always provide helpful and accurate responses based on the evaluation results.,
                   ➡️ tool_calls = None)

---

### (Optional) Advanced

In [19]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes